In [53]:
import numpy as np
import xarray as xr
import pandas as pd
import cftime
import dask
import matplotlib.pyplot as plt
import os
import xesmf as xe
import cesmesptools
import ocetrac as ot
import updated_tracker as ut
import pop_tools
from IPython.utils import io

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter
from cartopy.util import add_cyclic_point
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import matplotlib.dates as mdates

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter as fw
from matplotlib.animation import PillowWriter as pw

from IPython.utils import io
from os.path import exists
from tqdm import tqdm

# Open Dask Cluster

In [2]:
import dask_jobqueue
import distributed

# this first part is checking you're in the right environment
if "client" in locals():
    client.close()
    del client
if "cluster" in locals():
    cluster.close()

# this is where we set up the cluster, your own compute system if you will 
cluster = dask_jobqueue.PBSCluster(
    cores=1,  # The number of cores you want
    memory="15GB",  # Amount of memory
    processes=1,  # How many processes
    queue="casper",  # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    # log_directory="/glade/scratch/dcherian/dask/",  # Use your local directory
    resource_spec="select=1:ncpus=1:mem=15GB",  # Specify resources
    account="uwis0040",  # Input your project ID here / THIS WILL BE DIFFERENT FOR YOU 
    walltime="00:20:00",  # Amount of wall time
    interface="ext",  # Interface to use
)

# this is where we say that we want several of these compute systems,
# because we will have to deal with lots of data and can't just rely on one
cluster.adapt(maximum_jobs=24, minimum_jobs=1) # If you want to force everything to be quicker, 
# increase the number of minimum jobs, but sometimes then it will take a while until you get them assigned 
# (they have to queue), so it's a trade-off
client = distributed.Client(cluster)

# show the client that you have been assigned, you can click on the link and it will show you 
# a dashboard with all the tasks that have to be performed to do your calculation
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.108:38087,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Functions

In [3]:
def setup_axes(ax):
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=2)
    ax.coastlines(resolution='110m', color='black', lw=2)
    ax.set_ylabel('latitude')
    ax.set_xlabel('longitude')


def lons_to_360(data, coord='lon'):
    """ Converts longitude coordinates from (-180, 180) to (0, 360)."""
    data.coords[coord] = (360 + (data.coords[coord] % 360)) % 360
    data = data.sortby(data[coord])
    return data


def remove_trend(da, dim, deg=1):
    # detrend along a single dimension
    # return polyfit coefficients and detrended da
    p = da.polyfit(dim=dim, deg=deg, skipna=True)
    coord = da.coords[dim]
    fit = xr.polyval(coord, p.polyfit_coefficients)
    return da - fit


def get_anoms(da):
    clim = da.groupby('time.month').mean('time')
    da_noclim = da.groupby('time.month') - clim
    anoms = remove_trend(da_noclim, dim='time')
    return anoms


def regrid_SMYLE(ds, glat=1, glon=1):
    """
    Inputs:
        ds: xr.DataArray with coordinates that include TLAT and TLONG
    Returns:
        Regridded xr.DataArray with coordinates lat and lon
    """
    ds = ds.rename(({'TLONG': 'lon', 'TLAT': 'lat'}))
    ds_out = xe.util.grid_global(glon, glat)
    regridder = xe.Regridder(ds, ds_out, 'bilinear', periodic=True)
    regridded = regridder(ds)
    new_coords = regridded.assign_coords({'y': regridded.lat[:, 0].values, 'x': regridded.lon[0].values})
    return new_coords.drop_vars(['lat', 'lon']).rename({'x': 'lon', 'y': 'lat'})

# Load data

In [54]:
firstyear = 1989
lastyear = 2018
startmonth = 5
field = 'TEMP'

In [55]:
fold = '/glade/work/jtcohen'
fname = f'SMYLE_anom_drift_trend.{field}.{startmonth:02d}.{firstyear}-{lastyear}.nc'
smyle_ds = xr.open_dataset(f'{fold}/{fname}')
smyle_temp = smyle_ds['TEMP']

In [56]:
# Mask the Arctic, the Baltic Sea, the Red Sea, and the Black Sea
grid = pop_tools.get_grid('POP_gx1v7')
mask = xr.where((grid['REGION_MASK']>0) & (grid['REGION_MASK']<9), 1, np.nan)

In [57]:
# fig, ax = plt.subplots(1,1, figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree(200)})
# setup_axes(ax)
# im = ax.pcolormesh(grid.TLONG, grid.TLAT, mask, transform=ccrs.PlateCarree())
# plt.colorbar(im, ax=ax)
# plt.show()

In [58]:
smyle_masked = smyle_temp.where(mask==1, np.nan)

In [59]:
%%time
temp_1deg = regrid_SMYLE(smyle_masked)

CPU times: user 43.5 s, sys: 2.82 s, total: 46.4 s
Wall time: 49.2 s


In [60]:
temp_1deg.loc[dict(lat=slice(-90, -65))] = np.nan
temp_1deg.loc[dict(lat=slice(65, 90))] = np.nan

In [61]:
mask_1deg = ~np.isnan(temp_1deg.isel(Y=10, M=0, L=0))

In [62]:
# fig, ax = plt.subplots(1,1, figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree(200)})
# setup_axes(ax)
# ax.pcolormesh(mask_1deg.lon, mask_1deg.lat, mask_1deg, transform=ccrs.PlateCarree())

# Get threshold

At each point and each lead time, I take the 90th percentile over all the ensembles and initializations (start years).

In [63]:
threshold_val = 0.9

In [64]:
threshold = temp_1deg.quantile(threshold_val, dim=['Y', 'M'])

/glade/work/jtcohen/envs/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1583: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,


# Define features

In [65]:
smyle_features = temp_1deg.where(temp_1deg>=threshold, np.nan)

# Applying Ocetrac to SMYLE

## Ocetrac Code

In [67]:
def run_ocetrac(SST_features, mask, radius_val, min_size_quartile_val):
    """
    Run ocetrac on FOSI

    Inputs:
        SST: xr.DataArray with dimensions time, lat, lon
    Outputs:
        blobs: xr.DataArray with dimensions time, lat, lon
    """
    full_masked = SST_features.where(SST_features!=0)
    binary_out_afterlandmask=np.isfinite(full_masked)

    # Run Ocetrac
    xdim = 'lon'
    ydim = 'lat'
    Tracker = ut.Tracker(binary_out_afterlandmask[:,:,:], mask, radius=radius_val, min_size_quartile=min_size_quartile_val, timedim='L', xdim=xdim, ydim=ydim, positive=True)
    blobs = Tracker.track()
    return blobs


def apply_ocetrac_smyle(SST_features, mask, radius_val, min_size_quartile_val):
    """
    Run Ocetrac on SMYLE
    Inputs:
        SST: xr.DataArray with dimensions Y, M, L, lat, lon
    Outputs:
        blobs: xr.DataArray with dimensions time, lat, lon
    """
    with io.capture_output() as captured:
        blobs = SST_features.stack(ym=('Y', 'M')).groupby('ym').apply(
            run_ocetrac, args=(mask, radius_val, min_size_quartile_val)
        )
    return blobs.unstack()

# Iterate over radii and save

In [68]:
radius_vals = [1, 2, 4, 5, 6, 7]
min_size_quartile_val = 0

In [69]:
%%time
for radius_val in tqdm(radius_vals):
    blobs = apply_ocetrac_smyle(smyle_features, mask_1deg, radius_val, min_size_quartile_val)
    outdir = '/glade/work/jtcohen/'
    ds = xr.Dataset({'TEMP': temp_1deg, 'features': blobs})
    fout = f'SMYLE_features_premask.r{radius_val}.{field}.{startmonth:02d}.{firstyear}-{lastyear}.nc'
    if exists(outdir+fout): 
        print('File already exists. Are you sure you want to recalculate?')
    else:
        ds.load().to_netcdf(outdir+fout, engine='netcdf4')
        print(f'file saved at {outdir+fout}')
        ds.close()

 17%|█▋        | 1/6 [04:52<24:22, 292.43s/it]

file saved at /glade/work/jtcohen/SMYLE_features_premask.r1.TEMP.05.1989-2018.nc


 33%|███▎      | 2/6 [09:50<19:42, 295.73s/it]

file saved at /glade/work/jtcohen/SMYLE_features_premask.r2.TEMP.05.1989-2018.nc


 50%|█████     | 3/6 [16:04<16:34, 331.44s/it]

...

file saved at /glade/work/jtcohen/SMYLE_features_premask.r5.TEMP.05.1989-2018.nc


 83%|████████▎ | 5/6 [32:59<07:25, 445.44s/it]

file saved at /glade/work/jtcohen/SMYLE_features_premask.r6.TEMP.05.1989-2018.nc


100%|██████████| 6/6 [44:19<00:00, 443.31s/it]

file saved at /glade/work/jtcohen/SMYLE_features_premask.r7.TEMP.05.1989-2018.nc
CPU times: user 40min 13s, sys: 1min 41s, total: 41min 54s
Wall time: 44min 19s
